# Laboratorio 03 — Somnolencia del conductor: EAR, MAR, PERCLOS y motor de riesgo
**Curso:** Deep Learning y Agentes Inteligentes
**Docente:** Ing. Julián René Muñoz Burbano
**Proyecto:** DriveGuard Lab — Detección de somnolencia y riesgo del conductor

**Objetivo:** medir la apertura de ojos y boca con puntos faciales (MediaPipe), construir reglas para detectar
microsueño, cansancio y distracción, simular el protocolo de emergencia y evaluar el sistema con datos sintéticos
y grabaciones propias.

> **Aviso:** prototipo académico. No sustituye sistemas de seguridad vehicular certificados. Las pruebas se hacen
> con el vehículo detenido o frente al computador.

## 1. Arquitectura
```
Cámara → MediaPipe FaceLandmarker (478 puntos) → EAR · MAR · pose de la cabeza
       → [ViT de expresiones, en hilo aparte cada N s]
       → Motor de riesgo (reglas + calibración + histéresis)
       → Alarma local (pygame/winsound) + semáforo en Streamlit
       → Eventos → FastAPI → MongoDB
       → Protocolo de emergencia → correo / Telegram
```
**Cambios respecto a SentimentLab V3**

| V3 | DriveGuard Lab | Motivo |
|---|---|---|
| Haar Cascade (solo el rectángulo de la cara) | MediaPipe FaceLandmarker | Para medir somnolencia se necesitan los puntos de ojos y boca |
| ViT en el bucle de video | ViT en hilo aparte, cada `EXPRESION_INTERVALO_S` s | El ViT es lento en CPU; no debe frenar la detección |
| Sin sonido | Alarma local con pygame o winsound | El hilo de video de streamlit-webrtc no puede usar `st.audio` |
| — | Protocolo de emergencia | Una cámara no detecta choques: se usa "conductor que no responde", botón SOS y choque simulado |

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Directorio del proyecto:', ROOT)
print('Python:', sys.executable)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from core.config import ConfigRiesgo, CAPTURAS_DIR, VIDEOS_DIR, EJEMPLOS_DIR
from core.landmarks import calcular_ear, calcular_mar
from core.simulacion import ESCENARIOS, generar_mediciones, ejecutar
print('Escenarios sintéticos disponibles:', list(ESCENARIOS))

## 2. Eye Aspect Ratio (EAR)
Soukupová y Čech (2016) proponen medir la apertura del ojo con seis puntos:

$$EAR = \frac{\lVert p_2 - p_6 \rVert + \lVert p_3 - p_5 \rVert}{2\,\lVert p_1 - p_4 \rVert}$$

- $p_1$ y $p_4$: comisuras del ojo; $p_2, p_3$: párpado superior; $p_5, p_6$: párpado inferior.
- Ojo abierto: ≈ 0,25–0,35. Ojo cerrado: < 0,15. Es **invariante a la escala** (distancia a la cámara).
- En la malla de MediaPipe: ojo derecho `[33, 160, 158, 133, 153, 144]`, izquierdo `[362, 385, 387, 263, 373, 380]`.

In [ ]:
def ojo(apertura, ancho=30.0):
    a = apertura / 2
    return np.array([[0, 0], [10, -a], [20, -a], [ancho, 0], [20, a], [10, a]], float)

fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
for ax, ap, titulo in zip(axes, [10, 5, 1], ['Abierto', 'Entrecerrado', 'Cerrado']):
    p = ojo(ap)
    ax.fill(p[:, 0], -p[:, 1], alpha=.25)
    ax.plot(*np.c_[p.T[0], -p.T[1]].T, 'o')
    for i, (x, y) in enumerate(p, 1):
        ax.annotate(f'p{i}', (x, -y), textcoords='offset points', xytext=(3, 3))
    ax.set_title(f'{titulo}: EAR = {calcular_ear(p):.3f}'); ax.set_aspect('equal'); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Mouth Aspect Ratio (MAR) y bostezos
$$MAR = \frac{\frac{1}{3}\sum_{k=1}^{3} \lVert v_k^{sup} - v_k^{inf} \rVert}{\lVert c_{izq} - c_{der} \rVert}$$
Comisuras internas `78` y `308`; pares verticales `(81,178)`, `(13,14)`, `(311,402)`.
Boca cerrada ≈ 0–0,1; hablando ≈ 0,2–0,4; **bostezo > 0,6 durante ≥ 2 s**.

In [ ]:
comisuras = np.array([[0, 0], [40, 0]], float)
for apertura in [2, 10, 30]:
    v = [(np.array([20, -apertura / 2]), np.array([20, apertura / 2]))] * 3
    print(f'Apertura {apertura:>2} px -> MAR = {calcular_mar(comisuras, v):.2f}')

## 4. Indicadores y niveles de riesgo
| Indicador | Regla por defecto | Nivel |
|---|---|---|
| Microsueño | ojos cerrados de forma continua ≥ 1,5 s | **ALTO** |
| Cabeza caída | inclinación > 15° durante ≥ 3 s | **ALTO** |
| PERCLOS | > 30 % del último minuto con ojos cerrados | MEDIO |
| Bostezos | ≥ 3 en 5 minutos | MEDIO |
| Cabeceos | ≥ 3 en 60 s | MEDIO |
| Distracción | cabeza girada > 30° o rostro ausente > 3 s | MEDIO |
| Expresión negativa aparente | > 70 % de las clasificaciones en 30 s | BAJO |

- **Calibración:** los primeros 10 s se mide el EAR base del conductor; el umbral de ojo cerrado es el 75 % de ese valor.
- **Histéresis:** el nivel sube de inmediato, pero solo baja tras 2 s sin la condición (evita parpadeos de la alarma).
- **Conductor que no responde:** si el nivel es ALTO y durante 10 s más sigue con ojos cerrados, cabeza caída o
  sin rostro visible, se activa la emergencia (cuenta regresiva de 15 s y aviso a contactos).

In [ ]:
ConfigRiesgo()

## 5. Motor de riesgo con datos sintéticos
Cada escenario genera 15 mediciones por segundo con parpadeos normales cada 4 s.

In [ ]:
filas = []
orden = {'NORMAL': 0, 'BAJO': 1, 'MEDIO': 2, 'ALTO': 3}
for esc in ESCENARIOS:
    d = ejecutar(generar_mediciones(esc))
    d = d[~d.calibrando]
    motivos = sorted(set(','.join(d.motivos).split(',')) - {''})
    filas.append({'escenario': esc, 'nivel_max': max(d.nivel, key=orden.get),
                  'emergencia': d.emergencia.any(), 'perclos_max': round(d.perclos.max(), 2),
                  'bostezos': d.bostezos.max(), 'cabeceos': d.cabeceos.max(), 'motivos': ', '.join(motivos)})
pd.DataFrame(filas)

In [ ]:
def graficar(escenario, **kw):
    d = ejecutar(generar_mediciones(escenario), **kw)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    a1.plot(d.t, d.ear_crudo, lw=.6, alpha=.5, label='EAR crudo')
    a1.plot(d.t, d.ear, lw=1.2, label='EAR suavizado')
    a1.plot(d.t, d.ear_umbral, 'r--', label='umbral')
    a1.axvspan(0, d[d.calibrando].t.max(), color='tab:blue', alpha=.08, label='calibración')
    a1.set_ylabel('EAR'); a1.legend(loc='lower left', ncol=4); a1.set_title(f'Escenario: {escenario}')
    a2.step(d.t, d.nivel.map(orden), where='post', color='k')
    a2.fill_between(d.t, 0, 3.2, where=d.emergencia, color='red', alpha=.15, step='post', label='emergencia')
    a2.set_yticks(range(4), list(orden)); a2.set_xlabel('tiempo (s)'); a2.legend(loc='upper left')
    plt.tight_layout(); plt.show()
    return d

_ = graficar('microsueno')

In [ ]:
_ = graficar('sin_respuesta')

**Pregunta:** ¿en qué segundo se activa el nivel ALTO y en cuál la emergencia? Compárelo con la configuración (`microsueno_s`, `sin_respuesta_s`).

In [ ]:
_ = graficar('sin_respuesta', reconocer_en=25.0)   # el conductor pulsa "Estoy bien" a los 25 s

In [ ]:
d = ejecutar(generar_mediciones('cansancio'))
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(d.t, d.perclos * 100, label='PERCLOS (%)')
ax.axhline(30, color='r', ls='--', label='umbral 30 %')
ax2 = ax.twinx(); ax2.step(d.t, d.bostezos, color='tab:orange', where='post', label='bostezos')
ax.set_xlabel('tiempo (s)'); ax.legend(loc='upper left'); ax2.legend(loc='upper right')
ax.set_title('Escenario cansancio: parpadeos lentos y bostezos'); plt.show()

## 6. Experimento: ajuste de umbrales
Un buen sistema detecta los microsueños **sin** generar falsas alarmas en conducción normal. Simulamos 20
conductores distintos: cada uno tiene su propio EAR base (0,22–0,34), ruido de cámara y parpadeos de 0,15–0,7 s.
En el escenario de microsueño, el cierre dura entre 1,2 y 3 s. Variamos el umbral de duración del microsueño y el
factor del EAR y medimos el **porcentaje de microsueños detectados** y el de **conductores sin riesgo con falsa alarma**.

In [ ]:
def conductor(semilla):
    rng = np.random.default_rng(1000 + semilla)
    return dict(ear_base=rng.uniform(0.22, 0.34), ruido=rng.uniform(0.01, 0.035),
                dur_parpadeo=rng.uniform(0.15, 0.7), parpadeo_cada_s=rng.uniform(3, 6)), rng.uniform(1.2, 3.0)

N = 20
resultados = []
for factor in [0.65, 0.75, 0.85]:
    for micro in [0.5, 1.0, 1.5, 2.0]:
        cfg = ConfigRiesgo(factor_ear=factor, microsueno_s=micro)
        falsas = detectados = 0
        for s in range(N):
            params, cierre = conductor(s)
            n = ejecutar(generar_mediciones('normal', semilla=s, **params), cfg)
            falsas += (n.nivel == 'ALTO').any()
            m = ejecutar(generar_mediciones('microsueno', semilla=s, dur_cierre=cierre, **params), cfg)
            detectados += (m[m.t >= 20].nivel == 'ALTO').any()
        resultados.append({'factor_ear': factor, 'microsueno_s': micro,
                           'deteccion_%': 100 * detectados / N, 'falsas_alarmas_%': 100 * falsas / N})
tabla = pd.DataFrame(resultados)
tabla.pivot(index='microsueno_s', columns='factor_ear', values=['deteccion_%', 'falsas_alarmas_%'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for factor, g in tabla.groupby('factor_ear'):
    ax.plot(g['falsas_alarmas_%'], g['deteccion_%'], 'o-', label=f'factor {factor}')
    for _, f in g.iterrows():
        ax.annotate(f"{f.microsueno_s}s", (f['falsas_alarmas_%'], f['deteccion_%']), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Falsas alarmas (%)'); ax.set_ylabel('Microsueños detectados (%)'); ax.legend(); ax.grid(alpha=.3)
ax.set_title('Compromiso detección / falsas alarmas'); plt.show()

**Discusión:** un umbral de duración corto detecta todo, pero confunde parpadeos lentos con microsueños; uno largo
evita falsas alarmas, pero deja pasar cierres breves. Con personas reales influyen además la iluminación, las gafas,
la forma de los ojos y el ángulo de la cámara. Repita el experimento con sus propios videos (sección 8) y justifique
la configuración que elija.

## 7. Puntos faciales en una fotografía
Coloque una fotografía **autorizada** en `data/capturas/captura_prueba.jpg` (o tómela con la celda siguiente).
La carpeta se crea automáticamente; ya no ocurre el error del laboratorio 02.

In [ ]:
USE_CAMERA = False
foto = CAPTURAS_DIR / 'captura_prueba.jpg'
if USE_CAMERA:
    cap = cv2.VideoCapture(0); ok, frame = cap.read(); cap.release()
    if not ok: raise RuntimeError('No se pudo abrir la cámara')
    cv2.imwrite(str(foto), frame); print('Captura guardada en', foto)
print('Imagen disponible' if foto.exists() else f'Agregue una imagen en {foto} o active USE_CAMERA')

In [ ]:
from core.landmarks import (DetectorRostro, CONTORNO_OJO_DER, CONTORNO_OJO_IZQ, CONTORNO_BOCA,
                            OJO_DERECHO, OJO_IZQUIERDO)
if foto.exists():
    try:
        with DetectorRostro(modo='imagen') as det:
            img = cv2.imread(str(foto))
            r = det.procesar(img)
        if r is None:
            print('No se detectó rostro: mejore la iluminación o acérquese a la cámara')
        else:
            vis = img.copy()
            for x, y in r.puntos[:, :2].astype(int):
                cv2.circle(vis, (x, y), 1, (200, 200, 200), -1)
            for idx in (CONTORNO_OJO_DER, CONTORNO_OJO_IZQ, CONTORNO_BOCA):
                cv2.polylines(vis, [r.puntos[idx, :2].astype(np.int32)], True, (0, 255, 0), 1)
            for idx in OJO_DERECHO + OJO_IZQUIERDO:
                cv2.circle(vis, tuple(r.puntos[idx, :2].astype(int)), 3, (0, 0, 255), -1)
            plt.figure(figsize=(8, 6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
            pitch = f'{r.pitch:.1f}' if r.pitch is not None else '-'
            yaw = f'{r.yaw:.1f}' if r.yaw is not None else '-'
            print(f'EAR der={r.ear_der:.3f}  izq={r.ear_izq:.3f}  promedio={r.ear:.3f}  MAR={r.mar:.3f}  pitch={pitch}  yaw={yaw}')
            print('Blendshapes útiles:', {k: r.blendshapes.get(k) for k in ['eyeBlinkLeft', 'eyeBlinkRight', 'jawOpen']})
    except FileNotFoundError as exc:
        print(exc)
else:
    print('Prueba omitida: no existe imagen de entrada')

## 8. Análisis de un video propio
Grabe 60–90 s frente a la cámara: 10 s mirando al frente (calibración), parpadeos normales, ojos cerrados 3 s,
un bostezo largo, cabeceos y giros de cabeza. Guárdelo como `data/videos/prueba.mp4` o use la celda de grabación.

In [ ]:
GRABAR = False
SEGUNDOS = 60
video = VIDEOS_DIR / 'prueba.mp4'
if GRABAR:
    import time
    cap = cv2.VideoCapture(0)
    ancho, alto = int(cap.get(3)), int(cap.get(4))
    out = cv2.VideoWriter(str(video), cv2.VideoWriter_fourcc(*'mp4v'), 15, (ancho, alto))
    t0 = time.time()
    while time.time() - t0 < SEGUNDOS:
        ok, frame = cap.read()
        if not ok: break
        out.write(frame); time.sleep(1 / 15)
    cap.release(); out.release(); print('Video guardado en', video)
print('Video disponible' if video.exists() else f'No hay video en {video}')

In [ ]:
from core.pipeline import analizar_video
if video.exists():
    try:
        df = analizar_video(video, ConfigRiesgo(), salida=VIDEOS_DIR / 'prueba_anotado.mp4')
        v = df[~df.calibrando]
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
        a1.plot(v.t, v.ear, label='EAR'); a1.plot(v.t, v.ear_umbral, 'r--', label='umbral'); a1.legend()
        a2.step(v.t, v.nivel.map(orden), where='post'); a2.set_yticks(range(4), list(orden))
        plt.show()
        print('Rostro detectado en', f"{df.rostro.mean()*100:.0f}% de los fotogramas")
        print(v.motivos.replace('', np.nan).dropna().value_counts().head())
    except FileNotFoundError as exc:
        print(exc)
else:
    print('Análisis omitido: no hay video')

## 9. Expresiones faciales aparentes (ViT, opcional)
Se reutiliza el modelo de SentimentLab V3. Mida cuánto tarda en su equipo: si tarda más de 0,5 s por imagen,
suba `EXPRESION_INTERVALO_S` (p. ej. a 4 s) o desactívelo. **La detección de somnolencia no se ve afectada**, porque
el ViT corre en un hilo aparte. Recuerde: son *expresiones visibles*, no emociones internas.

In [ ]:
USE_VIT = False
if USE_VIT and foto.exists():
    import time
    from core.faces import analyze_frame
    img = cv2.imread(str(foto))
    analyze_frame(img)                     # primera llamada: carga/descarga del modelo
    t0 = time.perf_counter(); res = analyze_frame(img); dt = time.perf_counter() - t0
    print(f'Tiempo por imagen: {dt:.2f} s ->', 'aceptable' if dt < 0.5 else 'suba EXPRESION_INTERVALO_S')
    for rostro in res['rostros']:
        print(rostro['etiqueta'], rostro['confianza'])
else:
    print('Prueba del ViT desactivada (USE_VIT = False)')

## 10. Alarma local
El hilo de video de `streamlit-webrtc` no puede usar `st.audio`; por eso `core/alarma.py` reproduce el sonido en
el mismo equipo con un hilo propio: sirena continua en ALTO y aviso de dos tonos cada 6 s en MEDIO.

In [ ]:
SONAR = False
import time
from core.alarma import Alarma
alarma = Alarma(backend='auto' if SONAR else 'ninguno')
print('Reproductor:', alarma.backend)
for nivel in ['MEDIO', 'ALTO', 'NORMAL']:
    alarma.actualizar(nivel); time.sleep(1.5 if SONAR else 0.3)
alarma.detener()

## 11. Mensaje de emergencia
Configure correo y Telegram en `.env` (ver README). Con `ENVIAR = False` el envío es simulado.

In [ ]:
from core.notificaciones import construir_mensaje, notificar_contactos, estado_canales
conductor = {'nombre': 'Estudiante de prueba',
             'contactos': [{'nombre': 'Contacto 1', 'correo': 'contacto@example.com', 'telegram_chat_id': None}]}
asunto, cuerpo = construir_mensaje(conductor, 'sin_respuesta', 'ALTO',
                                   {'descripcion': 'Laboratorio de cómputo', 'lat': 1.2136, 'lon': -77.2811})
print(asunto); print(cuerpo)
print('Canales configurados:', estado_canales())
ENVIAR = False
print(notificar_contactos(conductor, 'prueba', simulado=not ENVIAR))

## 12. Simulación de impacto
Una cámara que mira al conductor no puede detectar un choque. En un vehículo real se usa un acelerómetro; aquí se
simula con un CSV (`t, ax, ay, az`). Un pico superior a 4 g se considera impacto.

In [ ]:
from core.impacto import cargar_acelerometro, detectar_impacto
fig, ax = plt.subplots(figsize=(11, 3))
for nombre in ['acelerometro_sin_choque.csv', 'acelerometro_choque.csv']:
    ruta = EJEMPLOS_DIR / nombre
    if not ruta.exists():
        from core.impacto import generar_csv_ejemplo
        generar_csv_ejemplo(ruta, choque='sin' not in nombre)
    acc = cargar_acelerometro(ruta)
    ax.plot(acc.t, acc.magnitud_g, label=nombre)
    print(nombre, detectar_impacto(acc))
ax.axhline(4, color='r', ls='--'); ax.set_ylabel('|a| (g)'); ax.set_xlabel('t (s)'); ax.legend(); plt.show()

## 13. MongoDB y API (opcional)
Inicie la API en otra consola: `python -m uvicorn backend.main:app --reload --port 8000`.

In [ ]:
import requests
from core.config import API_URL
try:
    print(requests.get(API_URL + '/health', timeout=3).json())
except requests.RequestException:
    print('La API no está iniciada; ejecute uvicorn desde la raíz del proyecto')

## 14. Actividades y evaluación
1. **Calibración personal:** registre su EAR base con y sin gafas, con buena y mala iluminación. ¿Cuánto cambia?
2. **Matriz de confusión de parpadeos:** grabe 2 minutos, anote a mano los instantes de ojos cerrados > 1,5 s y
   compare con la detección (precisión, recall, F1).
3. **Falsos positivos:** mirar el celular, reírse, hablar, usar gafas oscuras. ¿Qué reglas se activan? Proponga mejoras.
4. **Ajuste de umbrales:** repita el experimento de la sección 6 con sus videos y justifique la configuración elegida.
5. **Latencia:** mida FPS y tiempo desde que cierra los ojos hasta que suena la alarma.
6. **Protocolo de emergencia:** pruebe SOS, conductor que no responde y choque simulado; documente los mensajes recibidos.
7. **Ética:** consentimiento, privacidad (no se guardan imágenes) y límites de inferir emociones a partir del rostro.

**Entregables:** notebook ejecutado, tabla de métricas, capturas de la aplicación, mensajes de prueba recibidos y
un informe breve de análisis crítico de errores.